In [7]:
import math
import torch
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR


# 第一部分：制作克隆工具与因果掩码

# 深度复制N个完全一样的独立模块，用来叠加注意力网络
def clones(module, N):
    #ModuleList可以将复制好的网络注册到计算图中，可以反向传播，移动到GPU
    return nn.ModuleList([copy.deepcopy(module)for _ in range(N)])
    
# 生成下三角掩码
def subsequent_mask(size):
    attn_shape = (1, size, size)
    # 把不含对角线的上三角的数字变为1
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    # 返回一个布尔阵，True的位置将被注意
    return subsequent_mask == 0




    
# 第二部分：实现需要的注意力函数

# 缩放点积注意力"Scaled Dot-Product Attention"
def attention(query, key, value, mask=None, dropout=None):
    # 计算问询张量的长度 
    d_k = query.size(-1)
    # 缩放点积注意力公式
    # transpose(-2,-1)即为交换倒数第一个和数第二个维度
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # 处理mask,将为0的位置填入 -1e9，一般是放入pad掩码&因果掩码,使得注意力不被无意数据分散
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    # softmax归一化,行归一化
    p_attn = scores.softmax(dim = -1)

    # 归一化后考虑drop
    if dropout is not None:
        p_attn = dropout(p_attn)
    # 输出，p_attn记录了每一个词对其他词的关注程度，用于可视化数据，这里也做了一个接口返回
    return torch.matmul(p_attn, value), p_attn



# 多头注意力"MultiHeadAttention"
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        # 定义好要用的参数，后面传播时也要用上缩放点积注意力函数
        
        super(MultiHeadAttention, self).__init__()
        # 检查总特征维度，一般是词嵌入维度d_model能不能被h（头数）整除
        assert d_model % h == 0
        # 每个头分到特征维度
        self.d_k = d_model // h
        self.h = h
        # 创建4个投影矩阵，获得Q,K,V,O
        # 这里模型input = output = d_model，是一种工程优化，不是传统的低效的分出（d_model,d_k)
        # 然后分h个网络分别计算，最后合并。而是把h个头小矩阵在列方向拼成1个大矩阵
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        # 注意力分数
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            # 扩展mask维度适配注意力头（batch, 1, seq_len, seq_len)
            # 扩展后可以在所有头中复用
            mask = mask.unsqueeze(1)
        #获取输入数据的批次
        nbatches = query.size(0)
        #将Q,K,V三个（batch, seq_len, d_model)做linear(d_model, d_model),再view切成（batch,
        # seq_len, h, d_k),再转换为(batch, h, seq_len, d_k)，方便对每个头的数据做处理
        # for   in zip 就是第一轮就是 lin = linears[0](Wq), x = query,以此类推
        query, key, value = [lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
                             for lin, x in zip(self.linears,            (query, key, value))]
         # 调用缩放点积注意力 
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        # 多头拼接，先换回（batch, seq_len, h, d_k),transpose后内存物理排布不连续，
        # contiguous让内存重新连续，防止view报错，view再将后2个维度2合并为d_model  
        x = (x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k))
        # del 清理中间张量，释放显存
        del query
        del key
        del value
        # 使用最后一个线性层（Wo),综合输出融合
        return self.linears[-1](x)
                
 # 第三部分：前馈网络与位置编码(FFN&PE)


# 编写逐位置前馈网络"PositionwiseFeedForward"(FFN)
# 逐位置就是指每个位置完全独立计算，词之间互不干扰,就是一个简单的2层MLP
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        # 第一层，进入高维隐藏层(d_model -> d_ff)
        self.lin_1 = nn.Linear(d_model, d_ff)
        # 第二层，降维，转为正常输出特征
        self.lin_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.lin_2(self.dropout(self.lin_1(x).relu())) 

#编写词嵌入"Embeddings"
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        # lut为(Look-up Table)查找表的缩写，为（vocab, d_model)形状
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
    # 将(Batch, seq_len)转换成（Batch, seq_len, d_model)
    # * math.sqrt(self.d_model)防止词向量信息被位置编码淹没
    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

#编写位置编码,解决了自注意机制不能识别文字位置的问题
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        # 创建一个全0矩阵
        pe = torch.zeros(max_len, d_model)
        # 生成位置索引,(max_len, 1)
        position = torch.arange(0, max_len).unsqueeze(1)
        # 拿到三角函数中的分母倒数
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        # 广播机制算出数值，并填入pe
        # 偶数填sin
        pe[:, 0::2] = torch.sin(position * div_term)
        # 奇数填cos
        pe[:, 1::2] = torch.cos(position * div_term)
        # 添加一个维度(1, max_len, d_model)
        pe = pe.unsqueeze(0)
        # 注册为buffer，不会被优化器更新梯度，但可以转移到GPU中
        self.register_buffer("pe", pe)
    
    def forward(self,x):
        # 从pe中切分出合适大小，加入词特征维度中
        # x形状为（batch_size, seq_len, d_model)
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)



#第四步，通用残差规范化（layerNorm + Dropout + 残差相加）封装为通用外壳


#编写基础组件---层归一化
#把每一个词的特征向量强行拉回到“均值为 0、方差为 1”的标准正态分布，
#防止网络在层数很深时数值发生爆炸或弥散，让模型训练得更快、更稳定。
class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        # 缩放可学习参数
        self.a_2 = nn.Parameter(torch.ones(features))
        # 偏置可学习参数
        self.b_2 = nn.Parameter(torch.zeros(features))
        # 防止分母变为0的一个极小值
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

#编写残差连接外壳

#通用的残差连接外壳: x + Dropout(Sublayer(LayerNorm(x)))
# 先做norm再进入子层，称为Pre-LN，训练比post-LN更加稳定，不容易发散
# 这个包装类，可以简单的通过SublayerConnection(x, sublayer),
# 实现先norm归一化,再进入核心子层sublayer,之后dropout权重暂退训练，
# 最后实现残差连接的一体化处理。

class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

#第五步： 构建单层解码器与编码器"EncoderLayer&DecoderLayer"


# 编写单层编码器"EncoderLayer"
class EncoderLayer(nn.Module):
#编码器单层：包含 2 个子层 (Self-Attention 和 FFN)
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        # 创建2个包装，实现norm + 关键函数 + dropout + 残差分析
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size
    def forward(self, x, mask):
        # 自注意力层处理
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        # 前馈网络处理
        return self.sublayer[1](x, self.feed_forward)

#编码单层解码器"DecoderLayer"
class DecoderLayer(nn.Module):
#解码器单层：包含 3 个子层 (Masked Self-Attention, Cross-Attention 和 FFN)
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        # 词向量维度大小
        self.size = size
        # 掩码自注意力
        self.self_attn = self_attn
        # 交叉注意力
        self.src_attn = src_attn
        # 前馈全连接网络
        self.feed_forward = feed_forward
        # 3套快速一体式外壳
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        # 自身的掩码自注意力
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        # 交叉注意力
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        # 前馈网络
        return self.sublayer[2](x, self.feed_forward)


# 第六步：堆叠多层，实现真正编码器，解码器


#编写编码器
class Encoder(nn.Module):
#由 N 个 EncoderLayer 堆叠而成
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# 编写解码器
class Decoder(nn.Module):
    """由 N 个 DecoderLayer 堆叠而成"""
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

# 第7步：顶层封装与工厂


# 编写输出头"Generator"，将512维的特征向量翻译成输出的词
class Generator(nn.Module):
   """将隐藏层特征映射为目标词表概率分布: Linear + LogSoftmax"""
   def __init__(self, d_model, vocab):
       super(Generator, self).__init__()
# 1. 线性投影层：将维度从 d_model (如 512) 放大到 词表大小 (如 30000)
       self.proj = nn.Linear(d_model, vocab)

   def forward(self, x):
        # 使用log_softmax而不是softmax,取log，防止数值下溢（部分数值过小），同时交叉熵损失也要log
        # 通过 Log-Sum-Exp 技巧将 Softmax 和 Log 合并计算，不仅速度更快，而且数值更加精准稳定。
        return torch.nn.functional.log_softmax(self.proj(x), dim = -1)

#编写编码器解码器架构
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

#第八步：工厂装配函数（make_model)与初始化

#编写装配函数make_model，输出关键超参数，一键实现自动实例化，组装并初始化一个完整的transformer
def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    #定义好单份的多头注意力 attn、前馈网络 ff 和位置编码 position,之后可以快速克隆出独立新模块
    c = copy.deepcopy
    attn = MultiHeadAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )    
    #Xavier 均匀分布初始化权重 
    for p in model.parameters():
        #确保是二维权重矩阵，才做Xavier，一维直接全部设置0或1即可
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    return model

#第9步：数据与打包器

#编写Batch类，把一个批次（Batch）的原始句子打包、
#裁剪、打上遮罩，处理成模型训练的标准化格式。

class Batch:
    def __init__(self, src, tgt=None, pad=0):
        self.src = src
    #从(batch_size, src_len)变为(batch_size, 1, src_len)，
    #方便形状能自动匹配 (batch_size, num_heads, seq_len_q, seq_len_k)
        self.src_mask = (src != pad).unsqueeze(-2)
        if tgt is not None:
            # 2. 错位截取：输入和答案各错开一位
            self.tgt = tgt[:, :-1]
            self.tgt_y = tgt[:, 1:]
            # 组合掩码，因果掩码&padding掩码
            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            #统计token总数
            self.ntokens = (self.tgt_y != pad).data.sum()
    @staticmethod
    def make_std_mask(tgt, pad):
        # padding 掩码
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data)
        return tgt_mask                                      
        
#第10步：训练调度器与标签平滑（rate, LabelSmoothing, SimpleLossCompute)


#编写Noam学习率调度器(rate)
#前期：学习率从 0 开始慢慢线性爬升，像开车起步轻踩油门，让网络先平稳熟悉数据分布。
#中后期：学习率按步数的平方根倒数，缓慢下降，步子越迈越小，确保模型能精准滑向最优点，而不是在最优解附近反复横跳。
def rate(step, model_size, factor, warmup):
    if step == 0:
        step = 1
    return factor * (model_size ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5)))
"""
step：当前训练的总步数
model_size：模型的隐藏层维度
factor：全局缩放系数,用来整体调大/调小基准学习率
warmup：预热总步数
"""
# * (model_size ** (-0.5) 是因为特征维度越大的模型，在做矩阵相乘和残差累加时，神经元激发的总能量和梯度范数天然就更大。
# 模型维度越大，公式自动为你计算出更小的基准学习率，无需人工手动再去调小学习率，具有极强的自适应性。


#编写标签平滑函数（LabelSmoothing)
# 解决基于独热编码的softmax在多轮训练后拼命把权重放大，
#导致模型极度自信且固执（过拟合）。遇到生僻词或复杂语境时，缺乏容错和泛化能力。
# 原理：把一部分置信度抽出来，“施舍”平分给其他所有备选词：
class LabelSmoothing(nn.Module):
    def __init__(self, size, padding_idx, smoothing=0.0):
        super(LabelSmoothing, self).__init__()
    #采用 KL 散度来衡量预测分布与真实平滑分布之间的差异
    #专门用来计算“两个概率分布之间的差距”的数学工具。
    #选用sum而不是mean,mean会包含pad算平均损失，sum可以之后loss = sum / ntokens，准确
        self.criterion = nn.KLDivLoss(reduction ="sum")
    #记录pad对应的数字编号，防止把平滑概率分给pad,也免除pad带来的损失惩罚
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
    #词表的大小
        self.size = size
    def forward(self, x, target):
        #检查词表对不对
        assert x.size(1) == self.size
        # 复制一份与 x 形状相同的张量作为目标分布模板
        true_dist = x.data.clone()
         # 将平滑概率均分给其他词，正确词拿90%，pad不参与，所以-2
        true_dist.fill_(self.smoothing / (self.size - 2))
        #记录的标准答案位置（比如第 2 个词是正确答案），
        #直接把那个格子的数值从微小的强行替换成极高分
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        #把第 0 列一整列全部涂白擦掉，强制变回 0。
        true_dist[:, self.padding_idx] = 0
        # 当正确标签就是pad时，那就没必要计算它的损失，将词表分数全部清零
        # 进而在KL散度计算损失时，不会产生损失，实现完全忽略
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            #index_fill_ 把这些凑数行的所有格子通通抹成 0，这样模型就算在这些行上胡乱预测，
            #算出来的损失也是 0，彻底不干扰训练。
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        # KL散度计算差距，返回损失，detach()表明不用求导
        return self.criterion(x, true_dist.clone().detach())

# 编写结算函数SimpleLossCompute
# 将解码器输出的特征向量out，算出Loss，方便更新模型
class SimpleLossCompute:
    def __init__(self, generator, criterion):
        #把特征转成词表概率
        self.generator = generator
        # 打分算 KL 散度
        self.criterion = criterion


    def __call__(self, x, y, norm):
        #生成预测概率分布
        # x [batch_size, seq_len, vocab_size]
        # y [batch_size, seq_len]
        x = self.generator(x)
        # 拉直张量 + 算总分 + 平摊归一化
        sloss = (self.criterion(x.contiguous().view(-1, x.size(-1)), y.contiguous().view(-1))/ norm)
        # 双重返回：总损失数值, 带求导图的单步损失
        return sloss.data * norm, sloss



# 第11步，简单任务


# 编写数据生成器,生成src,tgt
def data_gen(V, batch_size, nbatches):
    for i in range(nbatches):
        data = torch.randint(1, V, size=(batch_size, 10))
        data[:, 0] = 1
        src = data.requires_grad_(False).clone().detach()
        tgt = data.requires_grad_(False).clone().detach()
        yield Batch(src, tgt, 0)

# 编写贪心算法，自回归式的贪心生成预测结果
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    # 提取记忆
    memory = model.encode(src, src_mask)
    #初始化生成序列，在初始化的序列中加入<BOS>
    ys = torch.zeros(1, 1).fill_(start_symbol).type_as(src.data)
    #循环max_len - 1次，循环生成预测词汇
    for i in range(max_len - 1):
        # 当ys = (1, L)
        # out = (1, L, d_model)
        out = model.decode(memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data))
        # 取最后一个位置特征，送入Generator预测词表分布
        #out[:, -1]是根据前面所有词提取的特征，是预测下一个词的关键
        prob = model.generator(out[:, -1])
        #选出概率最大的词
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        # 新词拼接在ys的末尾
        ys = torch.cat([ys, torch.zeros(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    return ys

#编写训练和验证的单论循环函数（run_epoch)
def run_epoch(data_iter, model, loss_compute, optimizer, scheduler, mode ="train"):
    total_tokens = 0
    total_loss = 0
    for i, batch in enumerate(data_iter):
        #向前传播
        out = model.forward(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        #计算loss, loss_node包含计算图，可用于反向传播
        loss, loss_node = loss_compute(out, batch.tgt_y, batch.ntokens)
        #训练模式
        if mode == "train":
            #算梯度
            loss_node.backward()
            #更新权重
            optimizer.step()
            #清空显存
            optimizer.zero_grad(set_to_none=True)
            #调整学习率
            scheduler.step()
        
        total_loss += loss
        total_tokens += batch.ntokens
    return total_loss / total_tokens, total_tokens


# 评估时的虚类的学习率调度器和更新器，复用epoch
class DummyOptimizer:
    def step(self):
        pass

    def zero_grad(self, set_to_none=False):
        pass


class DummyScheduler:
    def step(self):
        pass



# 简单样例
# 复述数字
def run_simple_example():
    #词表大小，0为padding, 1为Bos
    V = 11
    #设置平滑系数，设置为硬标签 
    criterion = LabelSmoothing(size = V, padding_idx = 0, smoothing =0.0)
    # 设置一个轻量的Transformer.快速验证结果
    model = make_model(V, V, N=2, d_model=32, d_ff=64, h=4)
    #设置优化器
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.5, betas = (0.9, 0.98), eps = 1e-9)
    #使用学习率调度器，绑定Noam调度公式（rate函数）
    lr_scheduler = LambdaLR(optimizer=optimizer,lr_lambda=lambda step: rate(
            step, model_size=model.src_embed[0].d_model, factor=1.0, warmup=400
        ),)
    #训练
    batch_size = 80
    for epoch in range(20):
        model.train()
    #生成20批数据
        run_epoch(data_gen(V, batch_size, 20),
              model,
              SimpleLossCompute(model.generator, criterion),
              optimizer,
              lr_scheduler,
              mode="train",
        )
        model.eval()
#[0]返回total_loss / total_tokens, total_tokens 中的前一个
        run_epoch(
        data_gen(V, batch_size, 5),
        model,
        SimpleLossCompute(model.generator, criterion),
        DummyOptimizer(),
        DummyScheduler(),
        mode="eval",
        )[0]

    model.eval()
    src = torch.LongTensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
    max_len = src.shape[1]
    src_mask = torch.ones(1, 1, max_len)
    print(greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=1))

    
run_simple_example()

tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]])
